# Saadat2021

[https://doi.org/10.3389/fpls.2021.750580](https://doi.org/10.3389/fpls.2021.750580)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Simulator, make_protocol, mca, plot, scan
from scipy.signal import peak_prominences

from mxlmodels import get_saadat2021

# This model needs Assimulo

## Helper Functions

In [ ]:
def npq(
    res_F,
    pam_prtc
):
    res_F_newindex = res_F.reset_index().rename(columns={"index": "time"}).copy()
    pulses = pam_prtc[pam_prtc["PPFD"] == pam_prtc["PPFD"].max()]
    pulses.index = pulses.index.total_seconds()
    res_extra = {
        "index": [],
        "Fm_time": [],
        "Fm": []
    }
    for i, time in enumerate(pulses.index):
        current_idx = (res_F_newindex["time"] - time).abs().idxmin()
        if i == 0:
            prior_idx = 0
        else:
            prior_idx = (res_F_newindex["time"] - pulses.index[i - 1]).abs().idxmin()
            prior_idx = current_idx - ((current_idx - prior_idx) // 2)
            
        if i == len(pulses.index) - 1:
            next_idx = len(res_F_newindex) - 1
        else:
            next_idx = (res_F_newindex["time"] - pulses.index[i + 1]).abs().idxmin()
            next_idx = current_idx + ((next_idx - current_idx) // 2)
            
        range_to_scan = res_F_newindex.iloc[prior_idx:next_idx]["Fluo"]

        res_extra["index"].append(range_to_scan.idxmax())
        res_extra["Fm_time"].append(res_F_newindex.loc[range_to_scan.idxmax(), "time"])
        res_extra["Fm"].append(range_to_scan.max())

    res_extra["NPQ"] = (res_extra["Fm"][0] - res_extra["Fm"]) / res_extra["Fm"]
    prominences, prominences_left, prominences_right = peak_prominences((res_F), res_extra["index"])
    
    res_extra["Fo"] = res_F.iloc[prominences_left].values
    res_extra["QY_PSII"] = (res_extra["Fm"] - res_extra["Fo"]) / res_extra["Fm"]
    
    return res_extra

# Figure 2

In [ ]:
length_period = 120
length_pulse = 0.8
pulse_intensity = 5000
dark_light = 40
actinic_light = 1000

length_nopulse = length_period - length_pulse

dark_period = [
    (length_nopulse, {"PPFD": dark_light}),
    (length_pulse, {"PPFD": pulse_intensity}),
]
light_period = [
    (length_nopulse, {"PPFD": actinic_light}),
    (length_pulse, {"PPFD": pulse_intensity}),
]

prtc = dark_period * 2 + light_period * 10 + dark_period * 9

fig2_prtc = make_protocol(prtc)

s = Simulator(get_saadat2021())
s.update_parameter("kf_cyclic_electron_flow", 0)

s.simulate_protocol(fig2_prtc)

fig2_res = s.get_result().unwrap_or_err().get_combined()

fig2_resextra = npq(
    res_F=fig2_res["Fluo"],
    pam_prtc=fig2_prtc
)

In [ ]:
fig2, ax = plt.subplots(figsize=(10, 5))

ax.plot(fig2_res["Fluo"] / max(fig2_res["Fluo"]), color="red", lw=2.5, label="Fluorescence")
ax.plot(fig2_resextra["Fm_time"], fig2_resextra["NPQ"], linestyle="dashed", color="black", label="NPQ", lw=2)

ax.legend(loc="upper right")

fig2_prtc_cleaned = fig2_prtc[fig2_prtc["PPFD"] != 5000]
prior = 0

for idx, row in fig2_prtc_cleaned.iterrows():
    if row["PPFD"] == fig2_prtc_cleaned["PPFD"].max():
        prior = idx.total_seconds()
        continue
    
    ax.axvspan(
        xmin=prior,
        xmax=idx.total_seconds(),
        color="#b3b3b3ff",
    )
    prior = idx.total_seconds()

ax.set(
    ylim=(0, 1.1),
    xlim=(0, 2500),
    xlabel="Time/(s)",
    ylabel="Fluorescence (normalised)",
)

plt.tight_layout()

plt.show()

# Figure 3

In [ ]:
fig3_res = scan.steady_state(
    model=get_saadat2021(),
    to_scan=pd.DataFrame({"PPFD": np.linspace(50, 1500, 500)})
)

fig3_res = pd.concat([fig3_res.variables, fig3_res.fluxes], axis=1)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10, 4))

for ax in axs:
    ax.set_xlabel(r"PPFD / (µmol $m^{-2}\ s^{-1})$")

# FIGURE A

axs[0].set_title("Photosynthetic Electron Fluxes")
axs[0].text(-0.15, 1.05, "A", transform=axs[0].transAxes, size=15, weight="bold")
secax = axs[0].twinx()

left_plots = {
    "PSI": {"res": fig3_res["PSI"], "color": "#1a75b5", "ls": "solid"},
    "LEF": {"res": fig3_res["PSII"] * 2, "color": "#fc7f09", "ls": "solid"},
    "CEF": {"res": fig3_res["cyclic_electron_flow"], "color": "#269b2a", "ls": "solid"},
    "Mehler": {"res": fig3_res["mehler"], "color": "#fa0002", "ls": "dashed"},
    "PTOX": {"res": fig3_res["PTOX"], "color": "#fa0002", "ls": "dotted"}
}

for label, vals in left_plots.items():
    if label in ["Mehler", "PTOX"]:
        ax = secax
    else:
        ax = axs[0]
    
    ax.plot(vals["res"], label=label, lw=2, color=vals["color"], ls=vals["ls"])

axs[0].set_ylabel("mmol e$^-$ / mol Chl / s")
axs[0].legend(loc="upper left")
    
secax.set_ylabel("mmol e$^-$ / mol Chl / s", color="red")
secax.tick_params(axis="y", colors="red")
secax.legend(loc="lower right")

# FIGURE B

axs[1].set_title("Energy and Redox Status")
axs[1].text(-0.15, 1.05, "B", transform=axs[1].transAxes, size=15, weight="bold")
secax = axs[1].twinx()

right_plots = {
    "ATP": {"res": fig3_res["ATP/tot"], "color": "#1270b1", "ls": "solid"},
    "NADPH": {"res": fig3_res["NADPH/tot"], "color": "#fa7d07", "ls": "solid"},
    "Fdred": {"res": fig3_res["Fd_ox/tot"], "color": "#259e24", "ls": "solid"},
    "PQred": {"res": fig3_res["PQ_ox/tot"], "color": "#9464c0", "ls": "solid"},
    "PCred": {"res": fig3_res["PC_ox/tot"], "color": "#895149", "ls": "solid"},
    "H2O2": {"res": fig3_res["H2O2"], "color": "#f80105", "ls": "dashed"}
}
for label, vals in right_plots.items():
    if label == "H2O2":
        ax = secax
    else:
        ax = axs[1]
    
    ax.plot(vals["res"], label=label, lw=2, color=vals["color"], ls=vals["ls"])

axs[1].set_ylabel("Fraction of the pool")
axs[1].legend(loc="upper left")
secax.set_ylabel("Concentration/mM", color="red")
secax.tick_params(axis="y", colors="red")
secax.legend(loc="lower right")

plt.tight_layout()

plt.show()

# Figure 4

In [ ]:
fig4_m = get_saadat2021()
fig4_m.update_parameter("PPFD", 1000)

num_points = 500
xvals_exp_part1 = np.linspace(-3, -1.75, num=num_points)
xvals_exp_part2 = np.linspace(-1.1, 3, num=num_points)

fig4_res = scan.steady_state(
    fig4_m,
    to_scan=pd.DataFrame({"kf_cyclic_electron_flow": fig4_m.get_parameter_values()["kf_cyclic_electron_flow"] * (2 **  np.concatenate([xvals_exp_part1, xvals_exp_part2]))}),
)

fig4_res = pd.concat([fig4_res.variables, fig4_res.fluxes], axis=1).dropna()
fig4_res.index = np.log2(fig4_res.index)

In [ ]:
fig, axs = plt.subplots(nrows=2, figsize=(8, 6), sharex=True)
# FIGURE A

axs[0].set_title("Photosynthetic Electron Fluxes")
axs[0].text(-0.15, 1.05, "A", transform=axs[0].transAxes, size=15, weight="bold")

secax = axs[0].twinx()

secax.set_ylabel("H$_2$O$_2$ / µM", color="red")
secax.tick_params(axis="y", colors="red")

left_plots = {
    "PSI": {"res": fig4_res["PSI"], "color": "#1a75b5"},
    "LEF": {"res": fig4_res["PSII"] * 2, "color": "#fc7f09"},
    "CEF": {"res": fig4_res["cyclic_electron_flow"], "color": "#269b2a"},
    "H2O2": {"res": fig4_res["H2O2"], "color": "#fa0002"},
}

for label, vals in left_plots.items():
    if label == "H2O2":
        ax = secax
    else:
        ax = axs[0]
    
    ax.plot(vals["res"], label=label, lw=2, color=vals["color"])
    
axs[0].set_ylabel("mmol e$^-$ / mol Chl / s")
axs[0].legend(loc="upper left")
secax.legend(loc="lower right")
    
# FIGURE B

axs[1].set_title("Energy and Redox Status")
axs[1].text(-0.15, 1.05, "B", transform=axs[1].transAxes, size=15, weight="bold")

right_plots = {
    "ATP": {"res": fig4_res["ATP/tot"], "color": "#1f77b4"},
    "NADPH": {"res": fig4_res["NADPH/tot"], "color": "#ff7f0e"},
    "Fdred": {"res": fig4_res["Fd_ox/tot"], "color": "#2da02d"},
    "PQred": {"res": fig4_res["PQ_ox/tot"], "color": "#9467bd"},
    "PCred": {"res": fig4_res["PC_ox/tot"], "color": "#8c564b"},
}

for label, vals in right_plots.items():
    axs[1].plot(vals["res"], label=label, lw=2, color=vals["color"])
    
axs[1].set_xlabel("log$_2$-fold change in PRG5 activity")
axs[1].set_ylabel("Fraction of the pool")
axs[1].legend(loc="upper left")

plt.tight_layout()

plt.show()

# Figure 5

In [ ]:
fig5_prtc = make_protocol([(60, {"PPFD": 600}), (60, {"PPFD": 40})] * 5)

s = Simulator(get_saadat2021())
s.simulate_protocol(fig5_prtc)

wt_res = s.get_result().unwrap_or_err().get_combined()

s = Simulator(get_saadat2021())
s.update_parameter("kf_cyclic_electron_flow", 0)
s.simulate_protocol(fig5_prtc)
ko_res = s.get_result().unwrap_or_err().get_combined()

fig5_res = {"wt": wt_res, "ko": ko_res}

In [ ]:
fig5, axs = plt.subplots(ncols=2, figsize=(10, 4))

lw = 3
axs[0].text(-0.15, 1.05, "A", transform=axs[0].transAxes, size=15, weight="bold")
axs[0].plot(ko_res["Fd_ox/tot"], label = "PGR-5 KO", lw=lw, color="red")
axs[0].plot(wt_res["Fd_ox/tot"], label = "wild type", lw=lw, color="black")

axs[1].text(-0.15, 1.05, "B", transform=axs[1].transAxes, size=15, weight="bold")
axs[1].plot(wt_res["rubisco_carboxylase"], lw=lw, color="black")
axs[1].plot(ko_res["rubisco_carboxylase"], lw=lw, color="red")

for ax in axs:
    ax.set_xlabel("Time / (s)")
    prior = 0
    for idx, row in fig5_prtc.iterrows():
        if row["PPFD"] == fig5_prtc["PPFD"].max():
            prior = idx.total_seconds()
            continue
        ax.axvspan(
            xmin=prior,
            xmax=idx.total_seconds(),
            color="#b3b3b3ff",
        )
        prior = idx.total_seconds()

axs[0].set_ylabel("Fd Redox State")
axs[1].set_ylabel("Rate of RuBisCO / (mM / s)")

fig.legend(loc="lower center", ncols=2)

plt.tight_layout(w_pad=2)

plt.show()

# Figure 6

In [ ]:
from mxlpy import model


scan_params = {
    "PSII_total": "PS2",
    "PSI_total": "PS1",
    "kcat_b6f": "b$_6$f",
    "kf_cyclic_electron_flow": "PGR5",
    "kMehler": "Mehler",
    "kcat_rubisco_carboxylase": "RuBisCO",
    "kcat_fbpase": "FBPase",
    "kcat_SBPase": "SBPase",
    "kcat_mda_reductase_2": "MDAR",
    "kcat_dehydroascorbate_reductase": "DHAR"
}

res = {}
fig6_m = get_saadat2021()

for pfd_val in [100, 1000]:
    fig6_m.update_parameter("PPFD", pfd_val)
    new_res = mca.response_coefficients(
            fig6_m,
            to_scan=scan_params.keys(),
            displacement=0.01
        )
    
    res[pfd_val] = pd.concat([new_res.variables, new_res.fluxes], axis=0)

res = pd.concat(res, names=["PPFD"], axis=0)
res = res.rename(columns=scan_params)


In [ ]:
ppfd100 = res.loc[100]
ppfd1000 = res.loc[1000]

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 7))

new_vars_pointer = {
    "Plastoquinone (reduced)": r"PQ$_{\mathrm{red}}$",
    "Ferredoxine (reduced)": r"Fd$_{\mathrm{red}}$",
    "Plastocyanine (reduced)": r"PC$_{\mathrm{red}}$",
    "NADPH": "NADPH",
    "ATP": "ATP",
    "RUBP": "RUBP",
    "3PGA": "PGA",
    "FBP": "FBP",
    "SBP": "SBP"
}

new_fluxes_pointer = {
    "PSII": "PS2",
    "PSI": "PS1",
    "b6f": "b$_6$f",
    "mehler": "Mehler",
    "cyclic_electron_flow": "CEF",
    "rubisco_carboxylase": "RuBisCO",
    "fbpase": "FBPase",
    "SBPase": "SBPase",
    "mda_reductase_2": "MDAR",
    "dehydroascorbate_reductase": "DHAR"
}

for i, df in enumerate([ppfd1000, ppfd100]):
    
    df_fluxes = df.loc[new_fluxes_pointer.keys()].copy()
    df_fluxes = df_fluxes.rename(index=new_fluxes_pointer)
    
    df_vars = df.loc[new_vars_pointer.keys()].copy()
    df_vars = df_vars.rename(index=new_vars_pointer)
    
    plot.heatmap(df_fluxes, ax=axs[i, 0])
    plot.heatmap(df_vars, ax=axs[i, 1])

for ax in axs.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.set_yticklabels(ax.get_yticklabels(), fontdict={"fontsize": 8})
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontdict={"fontsize": 8, "ha": "center"})
    
axs[0,0].set_title("Flux Control Coefficient")
axs[0,1].set_title("Concentration Control Coefficient")

fig.text(0.5, 1, r"PPFD = 1000 $\mathrm{\mu mol\ m^{-2}\ s^{-1}}$", ha="center", fontdict={"fontsize": 12})
fig.text(0.5, 0.49, r"PPFD = 100 $\mathrm{\mu mol\ m^{-2}\ s^{-1}}$", ha="center", fontdict={"fontsize": 12})

plt.tight_layout(h_pad=3)

plt.show()